In [2]:
import os
import os.path as op
from functools import partial

import matplotlib.pyplot as plt

import hnn_core
from hnn_core.params import _short_name

from hnn_core import duecker_ET_model, simulate_dipole, read_dipole, MPIBackend, average_dipoles
from hnn_core.optimization import Optimizer, add_opt_drives, set_params_opt_drives

import numpy as np

import pickle
hnn_core_root = op.join(op.dirname(hnn_core.__file__))
from scipy.stats import pearsonr

pathfig = os.path.join(os.getcwd(), 'figures')
pathgen = '/users/kduecker/hnn-tuning/hnn-data/'

from opt_ERP_cma import set_up_drives_empty

In [10]:
def get_constraints_init_params():

    with open('erp_supra_pred_params_cma.pkl', 'rb') as file:
        initial_params = pickle.load(file)


    constraints = dict({'prox1_L2_basket_ampa': (-15, -1.8),
                        'prox1_L2_pyramidal_ampa': (-5, 0),
                        'prox1_L5_basket_ampa': (-15, -1.75),
                        'prox1_L5ET_ampa': (-5, 0),

                        'prox1_L2_basket_nmda': (-15, -2.),
                        'prox1_L2_pyramidal_nmda': (-5, 0),
                        'prox1_L5_basket_nmda': (-15, -1.75),
                        'prox1_L5ET_nmda': (-5, 0),

                        'dist1_L2_basket_ampa': (-15, 0),
                        'dist1_L2_pyramidal_ampa': (-6, 0),
                        'dist1_L5ET_ampa': (-6, 0),

                        'dist1_L2_basket_nmda': (-15, 0),
                        'dist1_L2_pyramidal_nmda': (-6, 0),
                        'dist1_L5ET_nmda': (-6, 0),

                        'prox2_L2_basket_ampa': (-15, 0),
                        'prox2_L2_pyramidal_ampa': (-15, -1),
                        'prox2_L5_basket_ampa': (-15, 0),
                        'prox2_L5ET_ampa': (-15, 0),

                        'prox2_L2_basket_nmda': (-15, 0),
                        'prox2_L2_pyramidal_nmda': (-15, -1),
                        'prox2_L5_basket_nmda': (-15, 0),
                        'prox2_L5ET_nmda': (-15, 0),
                        
                        'prox1_mu': (15, 25),
                        'prox1_sigma': (1, 4),
                        'prox1_numspikes': (0, 3),
                        'dist1_mu': (50, 100),
                        'dist1_sigma': (1, 6),
                        'dist1_numspikes': (0,3),
                        'prox2_mu': (80, 150),
                        'prox2_sigma': (1, 20),
                        'prox2_numspikes': (0, 3)})

    return constraints, initial_params

In [11]:
def opt_cma(max_iter, popsize, n_procs, scaling_factor, smooth_win):
    data_url = f'{pathgen}MEG_detection_data/yes_trial_S1_ERP_all_avg.txt'
    exp_dpl = read_dipole(data_url)

    tstop = exp_dpl.times[-1]

    # get netwprl
    net_base = duecker_ET_model()
    # set up empty drives
    set_up_drives_empty(net_base)
    dt = 0.05           # increase dt to speed up 
    net_base._verbose = False

    constraints, initial_params = get_constraints_init_params()

    optim = Optimizer(net_base, tstop=tstop, constraints=constraints, solver='cma',
                        set_params=set_params_opt_drives, initial_params=initial_params, max_iter=max_iter, obj_fun="dipole_rmse_corr")

    optim.fit(target=exp_dpl, n_trials=3, scale_factor=scaling_factor,
            smooth_window_len=smooth_win, dt=dt, popsize=popsize, pth_backup='/users/kduecker/hnn-tuning/new_model_HNN/ERP', 
            bsl_cor="duecker", n_jobs=n_procs)


In [12]:
opt_cma(max_iter=5, popsize=20, n_procs=20, scaling_factor=1000, smooth_win=5)

(10_w,20)-aCMA-ES (mu_w=5.9,w_1=27%) in dimension 31 (seed=533113, Sun Apr  5 18:37:16 2026)
(20, 31)
[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.


--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.
--No graphics will be displayed.


[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:  3.9min
[Parallel(n_jobs=20)]: Done   2 out of  20 | elapsed:  4.1min remaining: 37.1min
[Parallel(n_jobs=20)]: Done   3 out of  20 | elapsed:  6.1min remaining: 34.6min
[Parallel(n_jobs=20)]: Done   4 out of  20 | elapsed:  6.6min remaining: 26.4min
[Parallel(n_jobs=20)]: Done   5 out of  20 | elapsed:  6.8min remaining: 20.4min
[Parallel(n_jobs=20)]: Done   6 out of  20 | elapsed:  6.8min remaining: 16.0min
[Parallel(n_jobs=20)]: Done   7 out of  20 | elapsed:  6.9min remaining: 12.8min
[Parallel(n_jobs=20)]: Done   8 out of  20 | elapsed:  7.1min remaining: 10.6min
[Parallel(n_jobs=20)]: Done   9 out of  20 | elapsed:  7.1min remaining:  8.7min
[Parallel(n_jobs=20)]: Done  10 out of  20 | elapsed:  7.2min remaining:  7.2min
[Parallel(n_jobs=20)]: Done  11 out of  20 | elapsed:  7.3min remaining:  5.9min
[Parallel(n_jobs=20)]: Done  12 out of  20 | elapsed:  7.3min remaining:  4.8min
[Parallel(n_jobs=20)]: Done  13 out of  20 | el

KeyboardInterrupt: 